# Validation E — Akerboom silica emitters

This notebook runs the six thermal cases in one multi-case YAML and compares the paper-stated nonradiative coefficient with a fitted effective coefficient. The three live S4 optics cases are optional because they are much more expensive.

**Learning goals:** execute selected cases from one YAML; separate paper inputs from calibration; recognize a model contradiction instead of hiding it with a fit.

## 1. Prepare the temporary Colab runtime

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
import shlex
import subprocess
import sys

from IPython.display import Image, Markdown, display

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    raise RuntimeError("Open this notebook in Google Colab before running setup.")

PROJECT_DIR = Path("/content/radcoolpv-py")

def run_command(args: list[str], cwd: Path | None = None, capture: bool = False):
    print("$", shlex.join(args))
    return subprocess.run(
        args, cwd=cwd, check=True, text=True,
        capture_output=capture,
    )

if not PROJECT_DIR.exists():
    run_command([
        "git", "clone", "--depth", "1", "--branch", "main",
        "https://github.com/gsilvaoelker/radcoolpv-py.git",
        str(PROJECT_DIR),
    ])

run_command([
    sys.executable, "-m", "pip", "install", "--quiet", "--editable", ".",
], cwd=PROJECT_DIR)
os.chdir(PROJECT_DIR)

print("Repository:", PROJECT_DIR)

## 2. Run the six thermal cases

In [ ]:
from radcoolpv import config as config_module
from radcoolpv import pipeline

validation_file = PROJECT_DIR / "validations/validation E/validation.yaml"
cases = config_module.load_cases(str(validation_file))
thermal_cases = [cfg for cfg in cases if cfg.run.thermal]

contexts = {}
for cfg in thermal_cases:
    contexts[cfg.case_name] = pipeline.run(cfg)

rows = []
for name, context in contexts.items():
    rows.append((name, context.thermal.equil_temp))
table = "| Case | Calculated equilibrium |\n|---|---:|\n" + "\n".join(
    f"| `{name}` | {temperature:.2f} K |" for name, temperature in rows
)
display(Markdown(table))

## 3. Inspect the thermal comparison

In [ ]:
for name in [
    "cooling_paper_h6_bare",
    "cooling_paper_h6_silica_cylinders",
    "cooling_calibrated_bare",
    "cooling_calibrated_silica_cylinders",
]:
    figure_dir = Path(contexts[name].results_dir) / "figures"
    for figure in sorted(figure_dir.glob("*.png")):
        display(Markdown(f"**{name}**"))
        display(Image(filename=str(figure)))

## 4. Optional: run the three live S4 optics cases

Leave this disabled for the normal classroom run. The optical comparison is useful, but a fresh run still needs a stated S4 mode-convergence study before it can support a stronger claim.

In [ ]:
import importlib
import importlib.util

S4_DIR = Path("/content/S4")
S4_COMMIT = "9569f5e555b967a4324eb1ea593d0f9f40761a61"

def install_s4() -> None:
    """Build the supported phoebe-p/S4 revision in this Colab runtime."""
    if importlib.util.find_spec("S4") is not None:
        print("S4 is already importable.")
        return
    run_command(["apt-get", "-qq", "update"])
    run_command([
        "apt-get", "-qq", "install", "-y", "build-essential", "git",
        "libboost-all-dev", "libfftw3-dev", "liblapack-dev",
        "libopenblas-dev", "libsuitesparse-dev",
    ])
    if not S4_DIR.exists():
        run_command(["git", "clone", "https://github.com/phoebe-p/S4.git", str(S4_DIR)])
    run_command(["git", "checkout", S4_COMMIT], cwd=S4_DIR)
    run_command(["make", "-j2", "S4_pyext"], cwd=S4_DIR)
    importlib.invalidate_caches()
    import S4
    print("S4:", S4.__file__)

In [ ]:
RUN_LIVE_OPTICS = False

if RUN_LIVE_OPTICS:
    install_s4()
    for cfg in [case for case in cases if case.run.optics]:
        pipeline.run(cfg)
else:
    print("Skipped the three live S4 optics cases.")

## 5. Interpret the contradiction

With the paper-stated $h=6.0$ W/m²/K, the calculated bare, flat-silica, and cylinder temperatures are 415.4 K, 360.6 K, and 355.6 K, not the reported 360 K, 339 K, and 336 K. A single fitted $h=12.54$ W/m²/K gives 359.7 K, 340.1 K, and 337.5 K, but that is calibration. The zero-emitter check is decisive: the stated balance gives 434.67 K while the paper reports 366.5 K. Do not label the calibrated match an independent validation.

**Exercise:** copy the YAML, test one intermediate value of $h$, and plot equilibrium temperature versus $h$. Keep the paper-stated and fitted cases separately labelled.

**Reference:** S. Akerboom et al., *ACS Photonics* 9, 3831–3840 (2022), [doi:10.1021/acsphotonics.2c01389](https://doi.org/10.1021/acsphotonics.2c01389).